In [ ]:

import os
import zipfile as zp
import shutil
import rasterio as rio
from rasterio.merge import merge
from shapely.geometry import Point, Polygon
import geopandas as gpd
import numpy as np
import pandas as pd
import tarfile as tf
import glob
import time

In [ ]:
BASE_FOLDER=r'C:\Local\Desktop_previous\miscellaneous\Neha\YieldData\21Jan\kamareddy'

In [ ]:
data_loc=os.path.join(BASE_FOLDER,'sentinel2')
data_apar=os.path.join(BASE_FOLDER,'apar')

In [ ]:
def mosaic_image(file_list,base_name):

    to_mosaic=[]
    for f in file_list:
        src=rio.open(f)
        to_mosaic.append(src)
    mosaic,out_trans = merge(to_mosaic)
    out_meta = src.meta.copy()
    out_meta.update({"driver": "GTiff","height": mosaic.shape[1],
                     "width": mosaic.shape[2],"transform": out_trans,
                     "crs":src.crs})
    out_path=os.path.join(BASE_FOLDER,'output')
    if not os.path.exists(out_path):
        os.makedirs(out_path)
    out_fp=os.path.join(out_path,(base_name+'.tif'))
    with rio.open(out_fp, "w", **out_meta) as dest:
        dest.write(mosaic)
    return out_path

In [ ]:
dates=[]
for r,d,f in os.walk(data_apar):
    for fl in f:
        if fl.endswith('.tif'):
            x=(fl.split('-')[1])
            if x!='2':
                dates.append(fl.split('-')[1])

In [ ]:
datelist=list(set(dates))

for n in datelist:
    parameter='APAR-'+str(n)
    bandname=[]
    for r,d,f in os.walk(data_apar):
        for fl in f:
            if (parameter in fl) and fl.endswith('.tif'):
                bandname.append(os.path.join(r,fl))
    mosaic_image(bandname,parameter)
    print(parameter)

In [ ]:
params=['NDVI','NDWI','ndpi','NDTI','SAVI','EVI']
for p in params:
    for n in np.arange(1,5):
        parameter=p+str(n)
        bandname=[]
        for r,d,f in os.walk(data_loc):
            for fl in f:
                if (parameter in fl) and fl.endswith('.tif'):
                    bandname.append(os.path.join(r,fl))
        mosaic_image(bandname,parameter)
        print(parameter)